# Day 27 · 跑 DPO

**配套讲义**: [`days/day-27.md`](../days/day-27.md) ｜ **需要 GPU（云机器）**

从 **SFT 版本**（不是基座）出发跑 DPO，看到 `rewards/chosen − rewards/rejected` 的差值稳步上升，并在领域集上测出与 SFT 版本的差异。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w5.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 先 inspect 数据（不用 GPU）

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.train.dpo",
                    "--inspect", "--in", "data/processed/dpo_train.jsonl"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 2. 长度偏见自检

**这是 DPO 最隐蔽的坑**：如果 rejected 一律比 chosen 短，模型可能只学到「变啰嗦」。

In [ ]:
import json, statistics
from pathlib import Path

p = Path("../data/processed/dpo_train.jsonl")
if p.exists():
    rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    cl = [len(r["chosen"]) for r in rows]
    rl = [len(r["rejected"]) for r in rows]
    print(f"chosen   中位长度 {statistics.median(cl):6.1f}")
    print(f"rejected 中位长度 {statistics.median(rl):6.1f}")
    ratio = statistics.median(cl) / max(statistics.median(rl), 1)
    print(f"比值 {ratio:.2f}")
    if ratio > 1.8:
        print("⚠️  chosen 明显更长 —— 模型可能只学会『说长一点』，考虑 average_log_prob 归一化")
    else:
        print("✓ 长度分布还算平衡")
else:
    print("先跑 Day 26 造数据")

## 3. 训练完看这两条曲线

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.train.monitor",
                    "outputs/qwen25vl3b-cx-dpo-v0"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 验收清单

- [ ] `--inspect` 显示 chosen/rejected 的长度分布（**必须检查长度偏见**）
- [ ] 训练启动，`rewards/margins` 稳步上升、`rewards/accuracies` 趋向 1
- [ ] 在 `cx_eval_v1` 上跑出 DPO 版本 vs SFT 版本的对照（哪怕差异很小）
- [ ] 能说出「这次 DPO 有没有训过头」（依据是 margin 曲线还是别的？）

**卡住了？** 回看 [`days/day-27.md`](../days/day-27.md) 第五节「容易踩的坑」。

> **明天**：`days/day-28.md` —— 可验证奖励（本地可跑，进阶可选）